In [68]:
import pandas as pd
import numpy as np
import json

In [ ]:
# to get the Prolific ID, sub_ind and presented stimulus set and movie names
ext_data = pd.read_csv('./ses1_data/data_all_subs_forQC.csv',index_col=[0])[['PID','sub_ind','stimset_rows','presented_trialIDs_all']]
ext_data

In [ ]:
# exclude bad subs
final_subs = np.unique(pd.read_csv('./ses1_data/final_triallevel_data.csv',index_col=[0])['subID']) # array of data with only final good subs
ext_data = ext_data.loc[ext_data['sub_ind'].isin(final_subs),:].reset_index()
ext_data

In [107]:
# #sanity check about presented videos
# df_presented_movies_check = pd.read_csv('./ses1_data/final_triallevel_data.csv',index_col=[0])[['subID','movie']]
# df_presented_movies_check.groupby(['subID'])['movie'].agg(list).reset_index()
# for i,row in df_presented_movies_check.iterrows():
    


In [ ]:
# open stim indices (array of 500 * ntrials (70)). each sub got a random row from the 500/
with open('./ses1_data/stimSets_ratingsv.json', 'r') as file:
    stimSets = json.load(file)
stimSets = stimSets['file_ind'] 
len(stimSets),len(stimSets[0])

In [72]:
def get_ses2_stim(ses1_stim): 
    # return stim indices for session 2 (i.e., unused numbers from range(1,141))
    # ses1_stim - stim ind presented in session 1
    full_list = list(range(0, 140)) # 0-139 (both including)
    ses2_stim = [num for num in full_list if num not in ses1_stim]
    ses1_stim.sort()
    return ses1_stim, ses2_stim

In [73]:
ses1_stim_all = []
ses2_stim_all = []
for ses1_stim in stimSets:
    ses1_stim, ses2_stim = get_ses2_stim(ses1_stim)
    # print(len(ses1_stim), len(ses2_stim))
    if len(ses2_stim) != 70:
        raise Exception(f'len(ses2_stim)={len(ses2_stim)}')
    ses1_stim_all.append(ses1_stim)
    ses2_stim_all.append(ses2_stim)

In [ ]:
PID = []

stim_row = []
ses1_stim_presented = []
ses2_stim_presented = []

for d,row in ext_data.iterrows():
    stimSet_row = row['stimset_rows']
    ses1_stim_presented.append(ses1_stim_all[stimSet_row])
    ses2_stim_presented.append(ses2_stim_all[stimSet_row])

ext_data['ses1_stim'] = ses1_stim_presented
ext_data['ses2_stim'] = ses2_stim_presented
ext_data.rename(columns = {'presented_trialIDs_all':'ses1_trialIDs'},inplace=True)
ext_data

In [ ]:
# open stim indices (array of 500 * ntrials (70)). each sub got a random row from the 500/
with open('./stim_info/all_videos.json', 'r') as file:
    trialIDs_list = json.load(file)
trialIDs_list = np.array(trialIDs_list['varList'])
len(trialIDs_list)

In [ ]:
len(row['ses2_stim'])

In [98]:
ses2_trialIDs = [list(trialIDs_list[np.array(row['ses2_stim'])]) for  i,row in ext_data.iterrows()]
ext_data['ses2_trialIDs'] = ses2_trialIDs

In [ ]:
stimSet_dict = ext_data[['PID','ses2_stim']].set_index(['PID']).to_dict()['ses2_stim']
stimSet_dict

In [108]:
with open('./ses2_data/stimSet_ses2.json', 'w') as file:
    json.dump(stimSet_dict, file, indent=4)  # indent for pretty printing

In [112]:
subs_list = list(stimSet_dict.keys())

In [ ]:
len(subs_list)

In [115]:
import csv

In [120]:
# Writing to CSV
with open('./ses2_data/ses2_subs.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    for item in subs_list:
        writer.writerow([item])  # Each item in its own row, note the [item] to make it a list

In [ ]:
[i for i in subs_list if len(i) != 24]

In [ ]:
# finally, manually fixed errors in these PIDs above!